# 📊 Key Market Data - Structuration et Enrichissement

Ce notebook structure et enrichit les données de marché (Stock Data) pour alimenter le **Company Universe**.

**Sources :**
- `2025-08-15_composition_sp500.csv` : Composition S&P 500 (Weight, Price)
- `2025-09-26_stocks-performance.csv` : Métriques de performance (Market Cap, Revenue, EPS, FCF, etc.)

**Objectif :**
- Fusionner les deux CSV
- Structurer en JSON par ticker
- Préparer l'intégration dans Company Universe


## Étape 1 : Imports


In [2]:
import pandas as pd
import json
from pathlib import Path
from typing import Dict
import os


## Étape 2 : Charger et Fusionner les CSV


In [ ]:
# Charger composition S&P 500
composition_df = pd.read_csv('../../data/raw/2025-08-15_composition_sp500.csv')

# Convertir Weight et Price (format européen avec virgule)
if 'Weight' in composition_df.columns:
    composition_df['Weight'] = composition_df['Weight'].astype(str).str.replace(',', '.').astype(float)
if 'Price' in composition_df.columns:
    composition_df['Price'] = composition_df['Price'].astype(str).str.replace(',', '.').astype(float)

print(f"✅ Composition S&P 500 : {len(composition_df)} entreprises")
print(composition_df.head())


✅ Composition S&P 500 : 503 entreprises
   #         Company Symbol  Weight   Price
0  1          Nvidia   NVDA  0.0765  181.99
1  2       Microsoft   MSFT  0.0667  520.99
2  3      Apple Inc,   AAPL  0.0597  233.28
3  4          Amazon   AMZN  0.0427  232.14
4  5  Meta Platforms   META  0.0339  783.78


In [ ]:
# Charger métriques de performance
performance_df = pd.read_csv('../../data/raw/2025-09-26_stocks-performance.csv')

print(f"✅ Métriques de performance : {len(performance_df)} entreprises")
print(performance_df.head())
print("\n📋 Colonnes disponibles:", performance_df.columns.tolist())


✅ Métriques de performance : 5508 entreprises
  Symbol           Company Name    Market Cap       Revenue    Op. Income  \
0   NVDA     NVIDIA Corporation  4.338392e+12  1.652180e+11  9.598100e+10   
1   MSFT  Microsoft Corporation  3.801767e+12  2.817240e+11  1.285280e+11   
2   AAPL             Apple Inc.  3.791126e+12  4.086250e+11  1.302140e+11   
3   GOOG          Alphabet Inc.  2.989536e+12  3.713990e+11  1.213700e+11   
4  GOOGL          Alphabet Inc.  2.981796e+12  3.713990e+11  1.213700e+11   

     Net Income    EPS           FCF  
0  8.659700e+10   3.52  7.202300e+10  
1  1.018320e+11  13.64  7.161100e+10  
2  9.928000e+10   6.59  9.618400e+10  
3  1.155730e+11   9.37  6.672800e+10  
4  1.155730e+11   9.38  6.672800e+10  

📋 Colonnes disponibles: ['Symbol', 'Company Name', 'Market Cap', 'Revenue', 'Op. Income', 'Net Income', 'EPS', 'FCF']


In [5]:
# Normaliser les colonnes pour la fusion
# Composition utilise 'Symbol', Performance aussi
if 'Symbol' in composition_df.columns:
    composition_df = composition_df.rename(columns={'Symbol': 'Ticker'})
if 'Symbol' in performance_df.columns:
    performance_df = performance_df.rename(columns={'Symbol': 'Ticker'})

# Fusionner sur Ticker
merged_df = pd.merge(
    composition_df[['Ticker', 'Company', 'Weight', 'Price']],
    performance_df,
    on='Ticker',
    how='outer',  # Garder tous les tickers
    suffixes=('_comp', '_perf')
)

print(f"✅ Fusionnée : {len(merged_df)} entreprises")
print("\n📊 Aperçu:")
print(merged_df.head(10))
print("\n📋 Colonnes finales:", merged_df.columns.tolist())


✅ Fusionnée : 5511 entreprises

📊 Aperçu:
  Ticker               Company  Weight   Price                   Company Name  \
0      A  Agilent Technologies  0.0006  118.81     Agilent Technologies, Inc.   
1     AA                   NaN     NaN     NaN              Alcoa Corporation   
2   AACB                   NaN     NaN     NaN     Artius II Acquisition Inc.   
3   AACG                   NaN     NaN     NaN          ATA Creativity Global   
4   AACI                   NaN     NaN     NaN    Armada Acquisition Corp. II   
5    AAL                   NaN     NaN     NaN   American Airlines Group Inc.   
6    AAM                   NaN     NaN     NaN   AA Mission Acquisition Corp.   
7   AAME                   NaN     NaN     NaN  Atlantic American Corporation   
8   AAMI                   NaN     NaN     NaN  Acadian Asset Management Inc.   
9   AAOI                   NaN     NaN     NaN  Applied Optoelectronics, Inc.   

     Market Cap       Revenue    Op. Income    Net Income   EPS   

## Étape 3 : Structurer en JSON par ticker


In [6]:
def create_market_data_structure(row: pd.Series) -> Dict:
    """
    Crée une structure JSON pour les données de marché d'une entreprise
    """
    data = {
        "ticker": str(row['Ticker']).upper() if pd.notna(row.get('Ticker')) else None,
        "company_name": str(row.get('Company', row.get('Company Name', ''))).strip() if pd.notna(row.get('Company', row.get('Company Name'))) else None,
        "sp500_weight": float(row['Weight']) if pd.notna(row.get('Weight')) else None,
        "current_price": float(row['Price']) if pd.notna(row.get('Price')) else None,
        "market_cap": float(row['Market Cap']) if pd.notna(row.get('Market Cap')) else None,
        "revenue": float(row['Revenue']) if pd.notna(row.get('Revenue')) else None,
        "operating_income": float(row['Op. Income']) if pd.notna(row.get('Op. Income')) else None,
        "net_income": float(row['Net Income']) if pd.notna(row.get('Net Income')) else None,
        "eps": float(row['EPS']) if pd.notna(row.get('EPS')) else None,
        "free_cash_flow": float(row['FCF']) if pd.notna(row.get('FCF')) else None,
    }
    
    # Calculer des ratios supplémentaires
    if data['revenue'] and data['net_income']:
        data['profit_margin'] = round(data['net_income'] / data['revenue'], 4)
    else:
        data['profit_margin'] = None
    
    if data['current_price'] and data['eps']:
        data['pe_ratio'] = round(data['current_price'] / data['eps'], 2) if data['eps'] != 0 else None
    else:
        data['pe_ratio'] = None
    
    if data['market_cap'] and data['revenue']:
        data['price_to_sales'] = round(data['market_cap'] / data['revenue'], 2) if data['revenue'] != 0 else None
    else:
        data['price_to_sales'] = None
    
    return data

# Appliquer à toutes les lignes
all_market_data = {}
for idx, row in merged_df.iterrows():
    ticker = str(row['Ticker']).upper() if pd.notna(row.get('Ticker')) else None
    if ticker:
        all_market_data[ticker] = create_market_data_structure(row)

print(f"✅ {len(all_market_data)} entreprises structurées")
print("\n📊 Exemple (AAPL):")
if 'AAPL' in all_market_data:
    print(json.dumps(all_market_data['AAPL'], indent=2, ensure_ascii=False))


✅ 5510 entreprises structurées

📊 Exemple (AAPL):
{
  "ticker": "AAPL",
  "company_name": "Apple Inc,",
  "sp500_weight": 0.0597,
  "current_price": 233.28,
  "market_cap": 3791126029400.0,
  "revenue": 408625000000.0,
  "operating_income": 130214000000.0,
  "net_income": 99280000000.0,
  "eps": 6.59,
  "free_cash_flow": 96184000000.0,
  "profit_margin": 0.243,
  "pe_ratio": 35.4,
  "price_to_sales": 9.28
}


## Étape 4 : Sauvegarder en JSON


In [ ]:
# Créer le dossier de sortie
output_dir = Path("../../data/generated/key_market_data")
output_dir.mkdir(exist_ok=True)

# Sauvegarder le fichier consolidé
output_file = output_dir / "all_market_data.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_market_data, f, indent=2, ensure_ascii=False)

print(f"✅ Fichier consolidé sauvegardé: {output_file}")
print(f"📊 Total: {len(all_market_data)} entreprises")

# Optionnel : Sauvegarder aussi en CSV pour visualisation
csv_file = output_dir / "merged_market_data.csv"
merged_df.to_csv(csv_file, index=False)
print(f"✅ CSV sauvegardé: {csv_file}")


✅ Fichier consolidé sauvegardé: key_market_data\all_market_data.json
📊 Total: 5510 entreprises
✅ CSV sauvegardé: key_market_data\merged_market_data.csv
